# Document Fraud / Tamper Detection Model (MFI)

This notebook trains and tests a **document anomaly detector** for loan application files (PDF/JPG/PNG).

## What it does
- Extracts binary-level features from uploaded documents
- Trains an `IsolationForest` on **normal** documents
- Creates synthetic tampered examples for testing
- Evaluates performance and saves model artifacts

> This is a v1 screening model for **manual review prioritization**, not final fraud adjudication.

In [ ]:
# If needed, uncomment and run:
# !pip install numpy pandas scikit-learn joblib matplotlib

from pathlib import Path
import hashlib
import math
import random
import shutil
import tempfile

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().resolve().parent
MEDIA_DOCS_DIR = PROJECT_ROOT / 'backend' / 'media' / 'loan_docs'
MODEL_DIR = PROJECT_ROOT / 'document_fraud_model'
ARTIFACT_PATH = MODEL_DIR / 'fraud_detector.joblib'

SUPPORTED_EXT = {'.pdf', '.jpg', '.jpeg', '.png'}
FEATURE_ORDER = [
    'size_kb',
    'entropy',
    'printable_ratio',
    'null_byte_ratio',
    'non_ascii_ratio',
    'unique_byte_ratio',
    'header_is_pdf',
    'header_is_jpeg',
    'header_is_png',
    'ext_is_pdf',
    'ext_is_jpeg',
    'ext_is_png',
    'ext_matches_header',
]

print('Project root:', PROJECT_ROOT)
print('Docs dir:', MEDIA_DOCS_DIR)
print('Artifact path:', ARTIFACT_PATH)

In [ ]:
def read_head(path: Path, max_bytes: int = 1024 * 1024) -> bytes:
    with path.open('rb') as f:
        return f.read(max_bytes)

def byte_entropy(data: bytes) -> float:
    if not data:
        return 0.0
    counts = np.bincount(np.frombuffer(data, dtype=np.uint8), minlength=256)
    probs = counts[counts > 0] / len(data)
    return float(-(probs * np.log2(probs)).sum())

def header_type(data: bytes) -> str:
    if data.startswith(b'%PDF-'):
        return 'pdf'
    if data.startswith(b'\xFF\xD8\xFF'):
        return 'jpeg'
    if data.startswith(b'\x89PNG\r\n\x1a\n'):
        return 'png'
    return 'unknown'

def ext_type(path: Path) -> str:
    ext = path.suffix.lower()
    if ext == '.pdf':
        return 'pdf'
    if ext in {'.jpg', '.jpeg'}:
        return 'jpeg'
    if ext == '.png':
        return 'png'
    return 'unknown'

def feature_map(path: Path) -> dict:
    data = read_head(path)
    size = path.stat().st_size
    ent = byte_entropy(data)

    if data:
        printable_ratio = sum(32 <= b <= 126 or b in (9, 10, 13) for b in data) / len(data)
        null_ratio = data.count(0) / len(data)
        non_ascii_ratio = sum(b > 127 for b in data) / len(data)
        unique_ratio = len(set(data)) / 256.0
    else:
        printable_ratio = null_ratio = non_ascii_ratio = unique_ratio = 0.0

    h = header_type(data)
    e = ext_type(path)

    return {
        'path': str(path),
        'size_kb': round(size / 1024.0, 4),
        'entropy': round(ent, 6),
        'printable_ratio': round(printable_ratio, 6),
        'null_byte_ratio': round(null_ratio, 6),
        'non_ascii_ratio': round(non_ascii_ratio, 6),
        'unique_byte_ratio': round(unique_ratio, 6),
        'header_is_pdf': 1.0 if h == 'pdf' else 0.0,
        'header_is_jpeg': 1.0 if h == 'jpeg' else 0.0,
        'header_is_png': 1.0 if h == 'png' else 0.0,
        'ext_is_pdf': 1.0 if e == 'pdf' else 0.0,
        'ext_is_jpeg': 1.0 if e == 'jpeg' else 0.0,
        'ext_is_png': 1.0 if e == 'png' else 0.0,
        'ext_matches_header': float(h == e and h != 'unknown'),
    }

def vectorize(df: pd.DataFrame) -> np.ndarray:
    return df[FEATURE_ORDER].to_numpy(dtype=np.float64)

In [ ]:
def collect_normal_docs(root: Path):
    docs = []
    if not root.exists():
        return docs
    for p in root.rglob('*'):
        if p.is_file() and p.suffix.lower() in SUPPORTED_EXT:
            docs.append(p)
    return docs

normal_docs = collect_normal_docs(MEDIA_DOCS_DIR)
print(f'Found {len(normal_docs)} real documents in media.')

if len(normal_docs) == 0:
    raise RuntimeError('No documents found in backend/media/loan_docs. Upload sample docs first.')

In [ ]:
def make_synthetic_tampered(source_docs, count=40):
    temp_dir = Path(tempfile.mkdtemp(prefix='tampered_docs_'))
    created = []

    picks = random.choices(source_docs, k=min(count, max(1, len(source_docs) * 2)))
    for idx, src in enumerate(picks):
        raw = src.read_bytes()
        mode = random.choice(['corrupt', 'header_swap', 'extension_mismatch'])

        if mode == 'corrupt':
            arr = bytearray(raw)
            flips = min(max(10, len(arr) // 200), 2000)
            for _ in range(flips):
                pos = random.randint(0, len(arr) - 1)
                arr[pos] = random.randint(0, 255)
            out = bytes(arr)
            ext = src.suffix.lower()
        elif mode == 'header_swap':
            ext = src.suffix.lower()
            out = b'NOTAREALHEADER' + raw[16:]
        else:  # extension_mismatch
            out = raw
            ext = random.choice(['.pdf', '.jpg', '.png'])
            if ext == src.suffix.lower():
                ext = '.pdf' if ext != '.pdf' else '.jpg'

        dest = temp_dir / f'tampered_{idx}{ext}'
        dest.write_bytes(out)
        created.append(dest)

    return temp_dir, created

tmp_dir, tampered_docs = make_synthetic_tampered(normal_docs, count=60)
print('Synthetic tampered docs:', len(tampered_docs))
print('Temp folder:', tmp_dir)

In [ ]:
normal_df = pd.DataFrame([feature_map(p) for p in normal_docs])
normal_df['label'] = 0

tampered_df = pd.DataFrame([feature_map(p) for p in tampered_docs])
tampered_df['label'] = 1

dataset = pd.concat([normal_df, tampered_df], ignore_index=True)
print('Dataset shape:', dataset.shape)
print(dataset[['label']].value_counts())

# Train only on normal docs (unsupervised anomaly detection)
train_norm, test_norm = train_test_split(normal_df, test_size=0.3, random_state=42)
test_mix = pd.concat([test_norm, tampered_df], ignore_index=True).sample(frac=1.0, random_state=42)

X_train = vectorize(train_norm)
X_test = vectorize(test_mix)
y_test = test_mix['label'].to_numpy()

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = IsolationForest(n_estimators=300, contamination=0.1, random_state=42)
model.fit(X_train_scaled)

decision = model.decision_function(X_test_scaled)
risk = 1 / (1 + np.exp(4.0 * decision))
pred = (risk >= 0.5).astype(int)

print('Confusion matrix:')
print(confusion_matrix(y_test, pred))
print('\nClassification report:')
print(classification_report(y_test, pred, digits=4))

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
artifact = {
    'model': model,
    'scaler': scaler,
    'feature_names': FEATURE_ORDER,
    'normal_train_count': int(len(train_norm)),
    'test_count': int(len(test_mix)),
}
joblib.dump(artifact, ARTIFACT_PATH)
print('Saved model artifact to:', ARTIFACT_PATH)

# Cleanup synthetic temp files
shutil.rmtree(tmp_dir, ignore_errors=True)
print('Temporary synthetic data removed.')

In [ ]:
def score_document(path: Path, artifact_path: Path = ARTIFACT_PATH):
    art = joblib.load(artifact_path)
    feats = feature_map(path)
    X = np.array([[feats[name] for name in art['feature_names']]], dtype=np.float64)
    Xs = art['scaler'].transform(X)
    decision = float(art['model'].decision_function(Xs)[0])
    risk = float(1.0 / (1.0 + math.exp(4.0 * decision)))

    score_100 = round(risk * 100, 2)
    if score_100 >= 70:
        level = 'high'
    elif score_100 >= 45:
        level = 'medium'
    else:
        level = 'low'

    return {
        'path': str(path),
        'risk_score': score_100,
        'risk_level': level,
        'decision_function': decision,
        'sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
    }

# Quick test on one real document
if normal_docs:
    sample = normal_docs[0]
    print(score_document(sample))